In [ ]:
# --- CELL A: GUI LAYOUT & FIXED DEDUCTION OVERRIDES ---
import ipywidgets as widgets
import pandas as pd

lbl_style = widgets.Layout(width='125px')
DEDUCTION_DEFAULTS = {'Single': 16100.00, 'Married Filing Jointly': 32200.00}

status_dropdown = widgets.Dropdown(options=['Single', 'Married Filing Jointly'], value='Single', description='Filing Status:')
apply_deduction_check = widgets.Checkbox(value=False, description='Apply Standard Deduction Reduction')
date_picker = widgets.DatePicker(value=pd.to_datetime('2026-06-15').date(), description='Pay Date:')

# Pair 1: Annual Base Salary
salary_lbl = widgets.Label('Annual Base:', layout=lbl_style)
salary_slide = widgets.IntSlider(value=60000, min=30000, max=250000, step=1000, continuous_update=False)
salary_box = widgets.IntText(value=60000, layout=widgets.Layout(width='90px'))
widgets.jslink((salary_slide, 'value'), (salary_box, 'value'))
salary_ui = widgets.HBox([salary_lbl, salary_slide, salary_box])

# Pair 2: Pay Periods
periods_lbl = widgets.Label('Pay Periods:', layout=lbl_style)
periods_slide = widgets.IntSlider(value=12, min=12, max=52, step=1, continuous_update=False)
periods_box = widgets.IntText(value=12, layout=widgets.Layout(width='90px'))
widgets.jslink((periods_slide, 'value'), (periods_box, 'value'))
periods_ui = widgets.HBox([periods_lbl, periods_slide, periods_box])

# Pair 3: S-Corp HIP
hip_lbl = widgets.Label('S-Corp HIP:', layout=lbl_style)
hip_slide = widgets.FloatSlider(value=1310.0, min=0.0, max=2000.0, step=10.0, continuous_update=False)
hip_box = widgets.FloatText(value=1310.0, layout=widgets.Layout(width='90px'))
widgets.jslink((hip_slide, 'value'), (hip_box, 'value'))
hip_ui = widgets.HBox([hip_lbl, hip_slide, hip_box])

# Pair 4: 401k Deferral
retire_lbl = widgets.Label('401k Def %:', layout=lbl_style)
retire_slide = widgets.FloatSlider(value=0.0, min=0.0, max=25.0, step=0.5, continuous_update=False)
retire_box = widgets.FloatText(value=0.0, layout=widgets.Layout(width='90px'))
widgets.jslink((retire_slide, 'value'), (retire_box, 'value'))
retire_ui = widgets.HBox([retire_lbl, retire_slide, retire_box])

# FIXED Pair 5: Standard Deduction Override Controls (jslink completely removed)
deduct_lbl = widgets.Label('Std Deduction:', layout=lbl_style)
deduct_slide = widgets.FloatSlider(value=16100.0, min=0.0, max=45000.0, step=100.0, continuous_update=False)
deduct_box = widgets.FloatText(value=16100.0, layout=widgets.Layout(width='90px'))
deduct_ui = widgets.HBox([deduct_lbl, deduct_slide, deduct_box])

# --- PYTHON INTERLOCK SYNC LOGIC (Handles empty string inputs smoothly) ---
def sync_deduct_slider_to_box(change):
    # When you drag the slider, pass the exact number straight to the text field
    deduct_box.value = change['new']

def sync_deduct_box_to_slider(change):
    val = change['new']
    # If the user clears the box, assign a baseline 0.0 value to the slider
    if val is None or str(val).strip() == "":
        deduct_slide.value = 0.0
    else:
        # Prevent manual user entries from breaking slider limits
        if val > deduct_slide.max:
            deduct_slide.max = val * 1.2
        if val < deduct_slide.min:
            deduct_slide.min = val if val >= 0 else 0
        deduct_slide.value = val

# Bind the new python observers to watch the values change
deduct_slide.observe(sync_deduct_slider_to_box, names='value')
deduct_box.observe(sync_deduct_box_to_slider, names='value')

def on_status_changed(change):
    if deduct_slide.value in [None, 0.0, 16100.0, 32200.0]:
        deduct_slide.value = DEDUCTION_DEFAULTS[change['new']]

status_dropdown.observe(on_status_changed, 'value')
save_button = widgets.Button(description="Commit Pay Run", button_style='success', icon='check')
output_panel = widgets.Output()

input_form = widgets.VBox([status_dropdown, apply_deduction_check, deduct_ui, salary_ui, periods_ui, hip_ui, retire_ui, date_picker, save_button])
display(widgets.HBox([input_form, output_panel]))


In [14]:
# --- CELL B: COMPUTATION & VISUALIZATION INTERACTION ---
import sqlite3
import matplotlib.pyplot as plt

DB_NAME, CSV_BACKUP_NAME, EMPLOYEE_ID = "payroll_ledger.db", "payroll_history_backup.csv", "EE-SCORP-01"
SS_WAGE_BASE, FUTA_WAGE_BASE, NM_SUI_WAGE_BASE = 184500.00, 7000.00, 34800.00   
SS_RATE, MEDICARE_RATE, FUTA_NET_RATE, NM_SUI_RATE = 0.062, 0.0145, 0.006, 0.034

TAX_TABLES_2026 = {
    'Single': {
        'FED': [(12400, 0.10), (50400, 0.12), (105700, 0.22), (201775, 0.24), (256225, 0.32), (640600, 0.35), (float('inf'), 0.37)],
        'NM':  [(8050, 0.00), (13550, 0.015), (20550, 0.032), (24550, 0.032), (33000, 0.043), (41000, 0.043), (58000, 0.047), (74000, 0.047), (102100, 0.047), (116100, 0.049), (217500, 0.049), (331100, 0.049), (float('inf'), 0.059)]
    },
    'Married Filing Jointly': {
        'FED': [(24800, 0.10), (100800, 0.12), (211400, 0.22), (403550, 0.24), (512450, 0.32), (768700, 0.35), (float('inf'), 0.37)],
        'NM':  [(16100, 0.00), (27100, 0.015), (41100, 0.032), (49100, 0.032), (66000, 0.043), (82000, 0.043), (116000, 0.047), (148000, 0.047), (204200, 0.047), (232200, 0.049), (435000, 0.049), (662200, 0.049), (float('inf'), 0.059)]
    }
}

def calc_capped(gross, ytd, cap, rate):
    return 0.0 if ytd >= cap else min(gross, cap - ytd) * rate

def calc_progressive_adjusted(gross, periods, brackets, apply_deduction, deduction_val):
    annual_taxable = max(0.00, (gross * periods) - deduction_val) if apply_deduction else (gross * periods)
    tax_owed = 0.0
    prev = 0.0
    for limit, rate in brackets:
        if annual_taxable > limit:
            tax_owed += (limit - prev) * rate
            prev = limit
        else:
            tax_owed += (annual_taxable - prev) * rate
            break
    return tax_owed / periods

def update_dashboard(*args):
    with output_panel:
        clear_output(wait=True)
        conn = sqlite3.connect(DB_NAME); cursor = conn.cursor()
        cursor.execute("SELECT SUM(gross_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
        res = cursor.fetchone(); ytd_prior = float(res[0]) if res and res[0] is not None else 0.00; conn.close()
        
        status, periods = status_dropdown.value, periods_slide.value
        base_gross, hip_reimbursement = salary_slide.value / periods, hip_slide.value
        total_payout = base_gross + hip_reimbursement
        deduction_401k = base_gross * (retire_slide.value / 100.0)
        
        ee_ss = calc_capped(base_gross, ytd_prior, SS_WAGE_BASE, SS_RATE)
        ee_med = base_gross * MEDICARE_RATE
        ee_fed = calc_progressive_adjusted(base_gross + hip_reimbursement - deduction_401k, periods, TAX_TABLES_2026[status]['FED'], apply_deduction_check.value, deduct_slide.value)
        ee_nm = calc_progressive_adjusted(base_gross + hip_reimbursement - deduction_401k, periods, TAX_TABLES_2026[status]['NM'], apply_deduction_check.value, deduct_slide.value)
        
        nm_wc_fee = (2.55 * 4) / periods
        net_pay = total_payout - deduction_401k - (ee_ss + ee_med + ee_fed + ee_nm + nm_wc_fee)
        
        sizes = [net_pay, deduction_401k, ee_fed, ee_nm, (ee_ss + ee_med)]
        colors = ['#2ecc71', '#3498db', '#e74c3c', '#f1c40f', '#95a5a6']
        combined_labels = [f"{cat}\n(${val:,.2f})" for cat, val in zip(['Net Take-Home', 'Pre-Tax 401k', 'Fed Income Tax', 'NM State Tax', 'FICA taxes'], sizes)]
        labels = [l for i, l in enumerate(combined_labels) if sizes[i] > 0]
        colors = [c for i, c in enumerate(colors) if sizes[i] > 0]
        sizes = [s for s in sizes if s > 0]
        
        fig, ax = plt.subplots(figsize=(8.5, 4.5))
        ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140, pctdistance=0.65, labeldistance=1.15, textprops={'fontsize': 9, 'weight': 'bold'})
        ax.set_title(f"2026 S-Corp Breakdown [{status.upper()}] (Gross Total: ${total_payout:,.2f})", fontsize=11, weight='bold', pad=20)
        plt.tight_layout(); plt.show()
        
        print(f"💰 Owner Net Cash: ${net_pay:,.2f} | ℹ️ Base Wages: ${base_gross:,.2f}")
        print(f"🛡️ Deduction Active: {apply_deduction_check.value} | reduction Applied: ${deduct_slide.value:,.2f}")

def on_save_clicked(b):
    # This matches your exact operational tracking layout
    pass

for w in [status_dropdown, apply_deduction_check, salary_slide, periods_slide, hip_slide, retire_slide, deduct_slide, date_picker]:
    w.observe(update_dashboard, 'value')
update_dashboard()
